In [1]:
                ### Travail réalisé par Mame Anta SARR et Ayaba Tabitha Favour GAGLOZOU ###

In [2]:
                                            ### Milestone M0 ###

In [4]:
import sys
from pathlib import Path

ROOT = Path.cwd()

while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from src.data import PATHS, SEED, describe_environment, load_raw, set_seed

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

print(f"seed = {set_seed(SEED)}")
print(describe_environment().to_string(index=False))

seed = 42
package version
 python  3.12.3
  numpy   2.5.2
 pandas   3.0.5
sklearn   1.9.0


In [5]:
df = load_raw()
print(f"shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head(3)

shape: 15,293 rows x 90 columns


,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,host_url,host_profile_id,host_profile_url,host_name,host_since,hosts_time_as_user_years,hosts_time_as_user_months,hosts_time_as_host_years,hosts_time_as_host_months,host_location,host_about,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,...,availability_365,calendar_last_scraped,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,availability_eoy,number_of_reviews_ly,estimated_occupancy_l365d,estimated_revenue_l365d,first_review,last_review,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,18674,https://www.airbnb.com/rooms/18674,20260624162917,2026-06-25,city scrape,Huge flat for 8 people close to Sagrada Familia,110m2 apartment to rent in Barcelona. Located ...,NaN,https://a0.muscache.com/pictures/hosting/Hosti...,71615,https://www.airbnb.com/users/show/71615,1.462508e+18,https://www.airbnb.com/users/profile/146250835...,Maria Y Mireia Barcelona4Seasons,NaN,16.0,5.0,15.0,5.0,"Barcelona, Spain","We are Mireia and Maria, two multilingual entr...",NaN,NaN,NaN,f,...,123,2026-06-25,59,9,1,121,9,54,22086.0,2013-05-27,2026-05-28,4.41,4.47,4.59,4.66,4.64,4.83,4.38,Spain - National registration number<br />ESFC...,NaN,20,20,0,0,0.37
1,23197,https://www.airbnb.com/rooms/23197,20260624162917,2026-06-25,city scrape,CCIB Forum· Large Balcony · 5 min walk to CCIB,"Beautiful, spacious apartment with large balco...",NaN,https://a0.muscache.com/pictures/hosting/Hosti...,90417,https://www.airbnb.com/users/show/90417,1.462509e+18,https://www.airbnb.com/users/profile/146250935...,Marnie,NaN,16.0,3.0,15.0,5.0,"Catalonia, Spain","Hi there,\n\nI’m marnie, originally from Austr...",NaN,NaN,NaN,t,...,239,2026-06-25,99,11,1,159,11,66,25608.0,2011-03-15,2026-06-07,4.84,4.95,4.91,4.94,4.99,4.69,4.71,Spain - National registration number<br />ESFC...,NaN,1,1,0,0,0.53
2,34981,https://www.airbnb.com/rooms/34981,20260624162917,2026-06-25,city scrape,VIDRE HOME PLAZA REAL on LAS RAMBLAS,Spacious apartment for large families or group...,NaN,https://a0.muscache.com/pictures/c4d1723c-e479...,73163,https://www.airbnb.com/users/show/73163,1.462508e+18,https://www.airbnb.com/users/profile/146250844...,Andres,NaN,16.0,5.0,15.0,5.0,"Barcelona, Spain","Hello I am a Professional designer, a traveler...",NaN,NaN,NaN,t,...,329,2026-06-25,304,37,1,159,33,222,88163.0,2010-10-03,2026-05-26,4.59,4.64,4.68,4.72,4.75,4.65,4.48,Spain - National registration number<br />ESFC...,NaN,2,2,0,0,1.59


In [6]:
# 1. One row represents:
print("rows:", f"{len(df):,}")
print("unique listing ids:", f"{df['id'].nunique():,}")
print("unique hosts:      ", f"{df['host_id'].nunique():,}")
print()
print("listings per host, top 5:")
print(df["host_id"].value_counts().head(5).to_string())

rows: 15,293
unique listing ids: 15,293
unique hosts:       4,595

listings per host, top 5:
host_id
346367515    588
1447144      448
21726991     359
32037490     293
4459553      243


In [7]:
# 2. Number of completely unusable columns (a number):
audit = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_pct": (df.isna().mean() * 100).round(1),
    "n_unique": df.nunique(dropna=True),
})

audit.sort_values("missing_pct", ascending=False).head(20)

,dtype,missing_pct,n_unique
neighborhood_overview,float64,100.0,0
host_since,float64,100.0,0
host_response_time,float64,100.0,0
host_thumbnail_url,float64,100.0,0
host_acceptance_rate,float64,100.0,0
host_response_rate,float64,100.0,0
host_verifications,float64,100.0,0
neighbourhood,float64,100.0,0
host_total_listings_count,float64,100.0,0
host_neighbourhood,float64,100.0,0


In [8]:
empty_cols = [c for c in df.columns if df[c].isna().all()]
constant_cols = [c for c in df.columns if df[c].nunique(dropna=True) <= 1]

print(f"{len(empty_cols)} columns are 100% empty:")
for c in empty_cols:
    print(f"    {c}")

print(f"\n{len(constant_cols)} columns are constant: {constant_cols}")
print(f"\nUsable columns: {df.shape[1] - len(set(empty_cols) | set(constant_cols))} of {df.shape[1]}")

12 columns are 100% empty:
    neighborhood_overview
    host_since
    host_response_time
    host_response_rate
    host_acceptance_rate
    host_thumbnail_url
    host_neighbourhood
    host_total_listings_count
    host_verifications
    neighbourhood
    calendar_updated
    instant_bookable

13 columns are constant: ['scrape_id', 'neighborhood_overview', 'host_since', 'host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_thumbnail_url', 'host_neighbourhood', 'host_total_listings_count', 'host_verifications', 'neighbourhood', 'calendar_updated', 'instant_bookable']

Usable columns: 77 of 90


In [9]:
# Step 1: find the relevant column.
candidates = [c for c in df.columns if "instant" in c.lower()]
print("candidate columns:", candidates)

# Step 2: is it usable?
col = "instant_bookable"
print(f"\n{col}:")
print(f"  dtype        : {df[col].dtype}")
print(f"  missing      : {df[col].isna().mean() * 100:.1f}%")
print(f"  unique values: {df[col].nunique(dropna=True)}")
print(f"  value counts :\n{df[col].value_counts(dropna=False).to_string()}")

# Step 3: the reply.
print("""Reply to Client B ----------------- We cannot answer this question with the current data. The snapshot contains an `instant_bookable` field, but it is empty for all 15,293 listings - the 
platform stopped publishing it in this data source. Nothing can be inferred about an instant-booking premium from this 
dataset. Two options: (a) we obtain the field from your own booking system, where it is recorded per listing; or (b) we 
treat this as out of scope and focus on the pricing model, which the available fields do support. We recommend (b) for now
and can revisit (a) if you can export the field.""")

candidate columns: ['instant_bookable']

instant_bookable:
  dtype        : float64
  missing      : 100.0%
  unique values: 0
  value counts :
instant_bookable
NaN    15293
Reply to Client B ----------------- We cannot answer this question with the current data. The snapshot contains an `instant_bookable` field, but it is empty for all 15,293 listings - the 
platform stopped publishing it in this data source. Nothing can be inferred about an instant-booking premium from this 
dataset. Two options: (a) we obtain the field from your own booking system, where it is recorded per listing; or (b) we 
treat this as out of scope and focus on the pricing model, which the available fields do support. We recommend (b) for now
and can revisit (a) if you can export the field.


In [12]:
                                        # A. Shape and grain
#Le dataset contient 15 293 lignes et 90 colonnes. Une ligne représente une annonce Airbnb et les 15 293 identifiants 
#d'annonces sont uniques. Cependant, les observations ne sont pas nécessairement indépendantes car 15 293 annonces 
#appartiennent à seulement 4 595 hôtes, certains hôtes possédant plusieurs annonces. Le plus grand hôte possède 588 
#annonces, ce qui doit être pris en compte lors de la séparation train/test et de la validation.

                                        # B. Completeness
audit[audit["missing_pct"] < 100].sort_values(
    "missing_pct", ascending=False
).head(3)

,dtype,missing_pct,n_unique
host_about,str,31.8,2406
bathrooms,float64,23.3,21
review_scores_location,float64,23.2,134


In [13]:
#Le dataset contient 12 colonnes entièrement vides, soit 100 % de valeurs manquantes. Parmi les colonnes qui sont 
#partiellement renseignées, les trois plus incomplètes sont host_about avec 31,8 % de valeurs manquantes, bathrooms avec 
#23,3 % et review_scores_location avec 23,2 %. Ces valeurs manquantes devront être prises en compte avant toute utilisation 
#de ces variables dans un modèle.

                                            # C. Types
df.dtypes.to_string()

#Deux types de variables nécessitent une attention particulière. La colonne price est stockée comme une chaîne de caractères
#(str) alors qu'elle représente une valeur numérique correspondant au prix d'une annonce; elle devra donc être nettoyée et 
#convertie avant une analyse numérique. De même, plusieurs variables comme first_review, last_review et last_scraped sont 
#stockées comme str alors qu'elles représentent des dates. Elles devraient être converties dans un format de date afin de 
#pouvoir effectuer correctement des opérations temporelles.

                                        # D. Questions
#Les données permettent d'étudier plusieurs questions:
#A. Comment le prix des annonces varie selon le type de logement?
#B. Comment les caractéristiques et les prix des annonces varient-ils selon les quartiers de Barcelone ?
#C. Comment la disponibilité des logements varie selon leur type? 

#En revanche, les données ne permettent pas de déterminer si les annonces avec réservation instantanée ont un prix plus 
#élevé, car la variable instant_bookable est entièrement vide dans ce snapshot. Cette question nécessiterait des données 
#complètes sur la réservation instantanée et un prix exploitable.

                                        # E. Problem statement
#CLIENT A: Pour la Direction du Tourisme de Barcelone, l'unité d'observation est une annonce Airbnb. L'objectif est 
#d'identifier les annonces susceptibles de fonctionner sans licence touristique valide afin d'aider les services 
#d'inspection à cibler leurs contrôles. La cible serait donc le statut de conformité de l'annonce. Le succès serait une 
#sélection d'annonces suffisamment fiable et défendable pour permettre aux services concernés de concentrer leurs 
#ressources d'inspection, plutôt que de simplement maximiser un score de performance du modèle.

#CLIENT B: Pour la société de gestion immobilière, l'unité d'observation est une annonce ou un appartement. L'objectif 
#est d'estimer le prix auquel une nouvelle annonce pourrait être proposée à partir de ses caractéristiques. La cible 
#serait donc le prix de l'annonce. Le succès serait d'obtenir une estimation suffisamment proche des prix pertinents du 
#marché pour aider l'entreprise à fixer un prix de mise en location, et non simplement d'obtenir une bonne performance sur 
#une métrique de régression.

'id                                                int64\nlisting_url                                         str\nscrape_id                                         int64\nlast_scraped                                        str\nsource                                              str\nname                                                str\ndescription                                         str\nneighborhood_overview                           float64\npicture_url                                         str\nhost_id                                           int64\nhost_url                                            str\nhost_profile_id                                 float64\nhost_profile_url                                    str\nhost_name                                           str\nhost_since                                      float64\nhosts_time_as_user_years                        float64\nhosts_time_as_user_months                       float64\nhosts_time_as_host_years      

In [14]:
                                                ### Milestone 1 ###